<a href="https://colab.research.google.com/github/Lucas66666677/Simple-Automatic-English-TA/blob/main/%E8%8B%B1%E6%96%87%E5%82%99%E8%AA%B2%E5%8A%A9%E6%89%8B%EF%BC%88Ollama%EF%BC%89%EF%BC%88Ollama%E7%94%9F%E6%88%90%E5%A4%AA%E6%85%A2%EF%BC%8C%E4%B8%8D%E5%BB%BA%E8%AD%B0%EF%BC%89.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1.安裝環境與相依套件

In [26]:
# 1. 安裝系統級 PDF 處理工具與解壓縮工具
!apt-get update -y
!apt-get install -y poppler-utils zstd pciutils

# 2. 安裝 Ollama 系統引擎
!curl -fsSL https://ollama.com/install.sh | sh

# 3. 安裝 Python 核心套件
!pip install -U langchain langchain-ollama gradio pypdf pdf2image requests
print("✅ 第一階段：系統與套件安裝完成！")

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 https://cli.github.com/packages stable InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
poppler-utils is already the newest v

# 2.Lecturas 主程式與 UI

In [32]:
import os
import time
import urllib.request
import io
import base64
import requests
import gradio as gr
import pypdf
import subprocess

# ==========================================
# 1. 啟動大腦 (強制綁定 GPU，拒絕 CPU 慢速運算)
# ==========================================
print("🛑 清理程序...")
os.system("pkill -9 ollama")
time.sleep(2)

print("🚀 啟動 Ollama 伺服器...")
# 【關鍵修改 1】強制指定 CUDA 顯示卡，確保它絕對是用 GPU 在算圖！
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"

log_file = open('/tmp/ollama_debug.log', 'w')
global ollama_server
ollama_server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=log_file,
    stderr=log_file
)

print("⏳ 等待大腦連線中...")
server_up = False
for i in range(20):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434', timeout=2)
        print("✅ 伺服器成功連線！")
        server_up = True
        break
    except:
        time.sleep(1)

if not server_up:
    print("❌ 伺服器崩潰！")
    raise Exception("伺服器啟動失敗")

print("⬇️ 確保模型已就緒...")
subprocess.run(["ollama", "pull", "llama3.2"])
# 【關鍵修改 2】換回官方最穩定的 llava 模型
subprocess.run(["ollama", "pull", "llava"])
print("🎉 系統啟動！準備就緒！")

# ==========================================
# 2. 核心功能
# ==========================================
TEMP_DIR = "/tmp/lecturas_temp"
os.makedirs(TEMP_DIR, exist_ok=True)

def encode_image_base64(pil_image):
    buffered = io.BytesIO()
    pil_image.save(buffered, format="JPEG")
    return base64.b64encode(buffered.getvalue()).decode('utf-8')

def process_lecturas(file_obj, text_input, image_obj):
    gr.Info("🤖 AI 正在高速運算中，請稍候...")
    print("\n" + "="*40)
    vision_output = ""
    clean_text = ""
    api_url = "http://127.0.0.1:11434/api/generate"

    if image_obj is not None:
        print("👉 發現圖片，正在壓縮並傳送給視覺大腦 (llava)...")
        try:
            import PIL.Image
            img = PIL.Image.open(image_obj) if isinstance(image_obj, str) else PIL.Image.fromarray(image_obj) if type(image_obj).__name__ == 'ndarray' else image_obj

            # 【關鍵修改 3】圖片極速壓縮，保證不當機、不卡頓
            img.thumbnail((800, 800))

            b64_img = encode_image_base64(img)
            payload = {
                "model": "llava",
                "prompt": "你是一位專業教師。請詳細解析這張圖片。請用繁體中文回答。",
                "images": [b64_img],
                "stream": False
            }
            start_time = time.time()
            res = requests.post(api_url, json=payload, timeout=120)
            end_time = time.time()
            print(f"✅ 圖片解析完成！極速耗時: {end_time - start_time:.2f} 秒")
            vision_output = f"### 📸 圖片解析結果\n{res.json().get('response', '')}\n\n---\n\n"
        except Exception as e:
            print(f"❌ 圖片解析發生錯誤: {e}")
            vision_output = f"⚠️ 圖片解析失敗: {e}\n\n"

    if file_obj:
        print("👉 正在讀取 PDF...")
        try:
            reader = pypdf.PdfReader(file_obj.name)
            for page in reader.pages:
                t = page.extract_text()
                if t: clean_text += t + "\n"
        except Exception as e:
            print(f"❌ PDF 讀取失敗: {e}")

    if not clean_text and text_input:
        clean_text = text_input

    final_left = ""
    final_right = vision_output

    if len(clean_text) >= 10:
        print("👉 正在生成板書與腳本 (llama3.2)...")
        try:
            prompt_a = f"<Role>資深教師。請用繁體中文回答。</Role><Task>判斷學科類型，整理結構化板書。</Task><Content>{clean_text}</Content>"
            res_a = requests.post(api_url, json={"model": "llama3.2", "prompt": prompt_a, "stream": False}, timeout=120)
            final_left = res_a.json().get('response', '生成失敗')

            prompt_b = f"<Role>補習班名師。請用繁體中文回答。</Role><Task>設計口語化教學腳本 (引發動機、觀念引導、實戰演練)。</Task><Content>{clean_text}</Content>"
            res_b = requests.post(api_url, json={"model": "llama3.2", "prompt": prompt_b, "stream": False}, timeout=120)
            final_right += res_b.json().get('response', '生成失敗')
            print("✅ 文字生成完成！")
        except Exception as e:
            print(f"❌ 文字生成發生錯誤: {e}")
            final_left = f"⚠️ 生成錯誤: {e}"
    else:
        if image_obj is not None:
             final_left = "💡 提示：您這次只有上傳圖片，沒有提供文本。解析結果請看右方欄位 👉"
        else:
             final_left = "⚠️ 無法讀取內容，請確認您有上傳檔案、文字或圖片。"

    print("🎉 任務全部結束！")
    print("="*40 + "\n")
    return final_left, final_right, "✅ 處理完成"

# ==========================================
# 3. 啟動網頁 UI
# ==========================================
with gr.Blocks(title="Lecturas v4.0 (Local Open Source)", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📖 Lecturas —— 備課助手 (v4.0 斷網開源版)")

    with gr.Row():
        with gr.Column(scale=1):
            file_input = gr.File(label="📄 1. 上傳教材 (PDF)", file_types=[".pdf"])
            text_input = gr.Textbox(label="或是貼上文字", lines=2)
        with gr.Column(scale=1):
            image_input = gr.Image(type="filepath", label="📸 2. 上傳圖片 (題目/文章/圖表)", height=200)

    btn = gr.Button("🚀 執行智慧分析", variant="primary")
    gr.Markdown("---")
    with gr.Row():
        out1 = gr.Markdown(label="板書 (依學科自動調整)")
        out2 = gr.Markdown(label="腳本與圖片詳解")
    with gr.Accordion("Debug Log", open=False):
        debug_out = gr.Textbox()

    btn.click(process_lecturas, inputs=[file_input, text_input, image_input], outputs=[out1, out2, debug_out])

demo.queue().launch(debug=True, share=True)

🛑 清理程序...
🚀 啟動 Ollama 伺服器...
⏳ 等待大腦連線中...
✅ 伺服器成功連線！
⬇️ 確保模型已就緒...
🎉 系統啟動！準備就緒！


/tmp/ipykernel_931/1896814350.py:138: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Lecturas v4.0 (Local Open Source)", theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3122907f13adeaa5df.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



👉 發現圖片，正在壓縮並傳送給視覺大腦 (llava)...
Keyboard interruption in main thread... closing server.
❌ 圖片解析發生錯誤: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
🎉 任務全部結束！

Killing tunnel 127.0.0.1:7862 <> https://3122907f13adeaa5df.gradio.live
